In [1]:
import pandas as pd
import searoute as sr
from geopy.distance import great_circle
import coordinates as coord
import os
from core_modules import core_modules as core

In this code, I assemble the necessary data, clean it, conduct an exploratory data analysis, and then ...

# Data Collection

## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [2]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is CO2 released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Instead of country names, country codes were used as per [country.txt](https://www.census.gov/foreign-trade/schedules/c/country.txt) file on Census.gov.

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

In [3]:
# gCO2/ton-km  - emission factor is CO2 released per ton of commodity per kilometer transported. Different countries use different methods

# ton_km = {
#     "china": 4,
#     "south_korea": 4.5, 
#     "vietnam": 7, 
#     "india": 6,
#     "mexico": 80 
# }

# ton_km = {
#     "5700": 4,
#     "5800": 4.5, 
#     "5520": 7, 
#     "5330": 6,
#     "2010": 80 
# }

## Distance

This code calculates distances between ports for countries beyond the ocean(China, India, South Korea, Vietnam), as well as land distance over the US border for Mexico.

### ISTHS6M

In [4]:
# first, analyze the files from census.gov site. They're encoded using fixed-width ASCII, which makes it hard for conventional Pandas functions to handle it
# for this reason, I had to enlist Claude's help to solve it.

# Correct colspecs for ISTHS6M (State HS6 Imports), record length 258
colspecs = [
    (0, 6),      # commodity (6-digit HS code)
    (6, 10),     # cty_code (4-digit country code)
    (10, 12),    # state (2-letter postal)
    (12, 16),    # year
    (16, 18),    # month
    (18, 33),    # gen_val_mo (general imports total value)
    (33, 48),    # con_val_mo (imports for consumption value)
    (48, 63),    # air_val_mo
    (63, 78),    # air_swt_mo (air shipping weight, kg)
    (78, 93),    # ves_val_mo
    (93, 108),   # ves_swt_mo (vessel shipping weight, kg)
    (108, 123),  # cnt_val_mo (containerized vessel value)
    (123, 138),  # cnt_swt_mo (containerized vessel weight)
    (138, 153),  # gen_val_yr (year-to-date)
    (153, 168),  # con_val_yr
    (168, 183),  # air_val_yr
    (183, 198),  # air_swt_yr
    (198, 213),  # ves_val_yr
    (213, 228),  # ves_swt_yr
    (228, 243),  # cnt_val_yr
    (243, 258),  # cnt_swt_yr
]
names = ['commodity', 'cty_code', 'state', 'year', 'month',
         'gen_val_mo', 'con_val_mo', 'air_val_mo', 'air_swt_mo',
         'ves_val_mo', 'ves_swt_mo', 'cnt_val_mo', 'cnt_swt_mo',
         'gen_val_yr', 'con_val_yr', 'air_val_yr', 'air_swt_yr',
         'ves_val_yr', 'ves_swt_yr', 'cnt_val_yr', 'cnt_swt_yr']

str_cols = ['commodity', 'cty_code', 'state', 'year', 'month']


### ISTHS6M (December 2024 — Pre-Tariff)

In [5]:
df_land_imports_2412 = pd.read_fwf(
    'data/original/isthsm2412.txt',
    colspecs=colspecs,
    names=names,
    dtype={c: str for c in str_cols},
)
df_land_imports_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,AK,2024,12,0,0,0,0,0,...,0,0,20337,20337,0,0,0,0,0,0
1,010121,1220,AR,2024,12,0,0,0,0,0,...,0,0,7500,7500,0,0,0,0,0,0
2,010121,1220,CA,2024,12,0,0,0,0,0,...,0,0,3632,3632,0,0,0,0,0,0
3,010121,1220,DE,2024,12,0,0,0,0,0,...,0,0,2183,2183,0,0,0,0,0,0
4,010121,1220,FL,2024,12,138066,138066,0,0,0,...,0,0,453343,453343,0,0,0,0,0,0


In [6]:
df_land_imports_2412.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194681 entries, 0 to 1194680
Data columns (total 21 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   commodity   1194681 non-null  object
 1   cty_code    1194681 non-null  object
 2   state       1194681 non-null  object
 3   year        1194681 non-null  object
 4   month       1194681 non-null  object
 5   gen_val_mo  1194681 non-null  int64 
 6   con_val_mo  1194681 non-null  int64 
 7   air_val_mo  1194681 non-null  int64 
 8   air_swt_mo  1194681 non-null  int64 
 9   ves_val_mo  1194681 non-null  int64 
 10  ves_swt_mo  1194681 non-null  int64 
 11  cnt_val_mo  1194681 non-null  int64 
 12  cnt_swt_mo  1194681 non-null  int64 
 13  gen_val_yr  1194681 non-null  int64 
 14  con_val_yr  1194681 non-null  int64 
 15  air_val_yr  1194681 non-null  int64 
 16  air_swt_yr  1194681 non-null  int64 
 17  ves_val_yr  1194681 non-null  int64 
 18  ves_swt_yr  1194681 non-null  int64 
 19  

### ISTHS6M (December 2025 — Post-Tariff)

In [7]:
df_land_imports_2512 = pd.read_fwf(
    'data/original/isthsm2512.txt',
    colspecs=colspecs,
    names=names,
    dtype={c: str for c in str_cols},
)
df_land_imports_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,CA,2025,12,25000,25000,0,0,0,...,0,0,25000,25000,0,0,0,0,0,0
1,010121,1220,CO,2025,12,6000,6000,0,0,0,...,0,0,12000,12000,0,0,0,0,0,0
2,010121,1220,CT,2025,12,0,0,0,0,0,...,0,0,3237,3237,0,0,0,0,0,0
3,010121,1220,FL,2025,12,164429,164429,0,0,0,...,0,0,251214,251214,0,0,0,0,0,0
4,010121,1220,ID,2025,12,0,0,0,0,0,...,0,0,22000,22000,0,0,0,0,0,0


This is the code that could be used to auto-import data. However, the problem with it is that there is poor connection between the API and the website.œ

In [8]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

### PORTHS6MM



In [9]:
# PORTHS6MM (Port HS6 Imports) record layout — record length 230
# Source: census.gov/foreign-trade/reference/products/layouts/dporths6i.html
colspecs = [
    (0, 6),      # commodity (6-digit HS code)
    (6, 10),     # cty_code (Schedule C, 4-digit)
    (10, 12),    # dist_unlade (Schedule D district, 2-digit)
    (12, 14),    # port_unlade (Schedule D port within district, 2-digit)
    (14, 18),    # year
    (18, 20),    # month
    (20, 35),    # gen_val_mo  (general imports value, this month)
    (35, 50),    # air_val_mo
    (50, 65),    # air_swt_mo  (air shipping weight, kg)
    (65, 80),    # ves_val_mo
    (80, 95),    # ves_swt_mo  (vessel shipping weight, kg)
    (95, 110),   # cnt_val_mo  (containerized vessel value)
    (110, 125),  # cnt_swt_mo  (containerized vessel weight)
    (125, 140),  # gen_val_yr  (year-to-date totals begin)
    (140, 155),  # air_val_yr
    (155, 170),  # air_swt_yr
    (170, 185),  # ves_val_yr
    (185, 200),  # ves_swt_yr
    (200, 215),  # cnt_val_yr
    (215, 230),  # cnt_swt_yr
]
names = ['commodity', 'cty_code', 'dist_unlade', 'port_unlade', 'year', 'month',
         'gen_val_mo', 'air_val_mo', 'air_swt_mo', 'ves_val_mo', 'ves_swt_mo',
         'cnt_val_mo', 'cnt_swt_mo',
         'gen_val_yr', 'air_val_yr', 'air_swt_yr', 'ves_val_yr', 'ves_swt_yr',
         'cnt_val_yr', 'cnt_swt_yr']
str_cols = ['commodity', 'cty_code', 'dist_unlade', 'port_unlade', 'year', 'month']


df_ocean_routes = pd.read_fwf(
    'data/original/PORTHS6MM2501.txt',
    colspecs=colspecs,
    names=names,
    dtype={c: str for c in str_cols},
)
df_ocean_routes["port_full"] = df_ocean_routes["dist_unlade"]+ df_ocean_routes["port_unlade"]


In [10]:
df_ocean_routes.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,port_full
0,010121,1220,07,08,2025,01,69084,0,0,0,...,0,0,69084,0,0,0,0,0,0,0708
1,010121,1220,09,01,2025,01,7500,0,0,0,...,0,0,7500,0,0,0,0,0,0,0901
2,010121,1220,33,10,2025,01,25000,0,0,0,...,0,0,25000,0,0,0,0,0,0,3310
3,010121,1220,38,01,2025,01,32236,0,0,0,...,0,0,32236,0,0,0,0,0,0,3801
4,010121,1220,38,02,2025,01,26500,0,0,0,...,0,0,26500,0,0,0,0,0,0,3802


https://www.census.gov/trade/downloads/2025/state_imp/hs6_m/ISTHSM2512.ZIP

## Weight of imports

Similar to UN Comtrade, but focuses on the USA and is more laconic.

[USA Trade Census](https://usatrade.census.gov/data/Perspective60/Browse/browsetables.aspx?utosid=5687ae9fc5be588295a68da15cd2f1cd&cache=tffv5e)
+ Data Source Selection: State Import Data(Harmonized System)
+ Filters:
    + Measures: Vessel SWT and Air SWT(kg) - The gross weight in kilograms of shipments made by seafaring vessel/airplane at customs
        + No data on land transportation there
    + State: All States
    + Commodity: 845011, 845012, 845019 and 845020(washing machines)
    + Country: India, South Korea, Mexico
    + Time: Jan 2025 - Mar 2026(monthly)

In [11]:
washing_machine_trade_filepath = "data/original/State Imports by HS Commodities_v4.csv"
washing_machine_df = pd.read_csv(washing_machine_trade_filepath, index_col=False, header=2)

In [12]:
washing_machine_df.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg),Unnamed: 5
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174,"1,777,528",NaN
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,NaN,"2,459,827",NaN
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,NaN,"2,268,490",NaN
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,NaN,"2,512,291",NaN
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,NaN,"2,541,012",NaN


# Data Cleaning
Cleaning is the longest and the most important step of any data cycle. After all, without good data there can be no good results. Because this projects uses data from a wide variety of sources, this means dealing with many different formats, which may complicate the cleaning process even further.

## Ocean-based Distance/Weight data(census.gov)
I have decided to address this data first, since it is more complete due to containing shipping weight, start/destination ports, as well as commodity codes.

In [13]:
asian_countries_codes = ["5700", "5800", "5520", "5330"]
df_asian_routes = df_ocean_routes[df_ocean_routes["cty_code"].isin(asian_countries_codes)]

In [14]:
df_asian_routes.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,port_full
108,010129,5800,39,01,2025,01,10000,10000,700,0,...,0,0,10000,10000,700,0,0,0,0,3901
170,010611,5520,10,12,2025,01,7920000,7920000,7360,0,...,0,0,7920000,7920000,7360,0,0,0,0,1012
171,010611,5520,54,01,2025,01,7500000,7500000,4284,0,...,0,0,7500000,7500000,4284,0,0,0,0,5401
287,010619,5330,17,04,2025,01,3500,3500,57,0,...,0,0,3500,3500,57,0,0,0,0,1704
288,010619,5700,04,17,2025,01,370924,370924,369,0,...,0,0,370924,370924,369,0,0,0,0,0417


In [15]:
df_asian_routes_v1 = df_asian_routes.T.drop_duplicates().T


In [16]:
df_asian_routes_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 102949 entries, 108 to 418323
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   commodity    102949 non-null  object
 1   cty_code     102949 non-null  object
 2   dist_unlade  102949 non-null  object
 3   port_unlade  102949 non-null  object
 4   year         102949 non-null  object
 5   month        102949 non-null  object
 6   gen_val_mo   102949 non-null  object
 7   air_val_mo   102949 non-null  object
 8   air_swt_mo   102949 non-null  object
 9   ves_val_mo   102949 non-null  object
 10  ves_swt_mo   102949 non-null  object
 11  cnt_val_mo   102949 non-null  object
 12  cnt_swt_mo   102949 non-null  object
 13  port_full    102949 non-null  object
dtypes: object(14)
memory usage: 15.8+ MB


In [17]:
df_asian_routes_v1.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,port_full
108,010129,5800,39,01,2025,01,10000,10000,700,0,0,0,0,3901
170,010611,5520,10,12,2025,01,7920000,7920000,7360,0,0,0,0,1012
171,010611,5520,54,01,2025,01,7500000,7500000,4284,0,0,0,0,5401
287,010619,5330,17,04,2025,01,3500,3500,57,0,0,0,0,1704
288,010619,5700,04,17,2025,01,370924,370924,369,0,0,0,0,0417


In [18]:
numerical_cols_asian = ["gen_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols_asian:
    df_asian_routes_v1[col] = pd.to_numeric(df_asian_routes_v1[col])
df_asian_routes_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 102949 entries, 108 to 418323
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   commodity    102949 non-null  object
 1   cty_code     102949 non-null  object
 2   dist_unlade  102949 non-null  object
 3   port_unlade  102949 non-null  object
 4   year         102949 non-null  object
 5   month        102949 non-null  object
 6   gen_val_mo   102949 non-null  int64 
 7   air_val_mo   102949 non-null  int64 
 8   air_swt_mo   102949 non-null  int64 
 9   ves_val_mo   102949 non-null  int64 
 10  ves_swt_mo   102949 non-null  int64 
 11  cnt_val_mo   102949 non-null  int64 
 12  cnt_swt_mo   102949 non-null  int64 
 13  port_full    102949 non-null  object
dtypes: int64(7), object(7)
memory usage: 15.8+ MB


In [19]:
df_asian_routes_wash = df_asian_routes_v1[df_asian_routes_v1["commodity"].isin(["845011", "845020"])]
df_asian_routes_wash.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,port_full
273372,845011,5520,10,03,2025,01,1476672,0,0,1476672,215096,1476672,215096,1003
273373,845011,5520,13,03,2025,01,3888,0,0,3888,740,3888,740,1303
273374,845011,5520,14,01,2025,01,8976,0,0,8976,2224,8976,2224,1401
273375,845011,5520,17,03,2025,01,153763,0,0,153763,21411,153763,21411,1703
273376,845011,5520,27,04,2025,01,19801,0,0,19801,4583,19801,4583,2704


This code is the proof of concept for how calculating sea route distances with the "searoute" model is going to work. The coordinates are [lon, lat], and this example shows a route from Shanghai to Los Angeles.

In [20]:
route = sr.searoute([121.47, 31.23], [-118.27, 33.74])
# in km, following real sea lanes
distance_km = route.properties['length']  

As there are only 76 rows in the final dataframe, it's possible to iterate over them by applying the distance-calculation function to each row.

In [21]:
# this is necessary, because the dataframe will keep getting modified.
df_asian_routes_wash_v1 = df_asian_routes_wash.copy()

In [22]:
inland_ports = {"3901", "4101", "5206"}
df_asian_routes_wash_v1 = df_asian_routes_wash_v1[
    ~df_asian_routes_wash_v1["port_full"].isin(inland_ports)
]
df_asian_routes_wash_v1.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,port_full
273372,845011,5520,10,03,2025,01,1476672,0,0,1476672,215096,1476672,215096,1003
273373,845011,5520,13,03,2025,01,3888,0,0,3888,740,3888,740,1303
273374,845011,5520,14,01,2025,01,8976,0,0,8976,2224,8976,2224,1401
273375,845011,5520,17,03,2025,01,153763,0,0,153763,21411,153763,21411,1703
273376,845011,5520,27,04,2025,01,19801,0,0,19801,4583,19801,4583,2704


In [23]:
route = sr.searoute(coord.asian_origins["5700"], coord.us_ports["2704"])
print(route)  # inspect the structure

{"geometry": {"coordinates": [[121.418678, 31.509996], [121.839752, 31.258596], [122.9, 31.3], [125.34383, 32.983736], [126.035156, 33.45436], [126.643151, 33.735423], [127.444583, 34.104419], [128.16925, 34.40691], [128.916321, 34.761923], [129.2, 35], [129.597473, 35.429344], [129.972285, 35.665533], [139.744224, 41.259182], [140.352539, 41.363637], [141.277588, 41.681992], [143.401627, 41.699651], [146, 43.2], [153.687693, 46.196119], [157.933307, 47.444876], [160.194187, 48.109865], [160.925037, 48.324829], [161.4373, 48.4755], [169.78575, 50.154218], [170.2039, 50.2383], [172.354557, 50.483925], [174.140093, 50.687849], [180, 50], [180, 50], [180.514157, 50.252896], [183.003777, 50.558732], [188.444846, 51.295103], [191.7944, 51.0966], [191.91903, 51.079944], [194.377043, 50.751454], [197.205832, 50.373414], [200, 50], [205.4646, 49.1714], [206.544495, 48.950813], [208.409701, 48.569813], [209.998947, 48.245182], [210.7234, 48.0972], [212.515975, 47.633164], [215.7457, 46.7971], [

In [24]:
df_asian_routes_wash_v1["distance"] = df_asian_routes_wash_v1.apply(
    core.compute_distance, args = (coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)

In [25]:
# def compute_distance_ocean(row):
#     origin = asian_origins[row["cty_code"]]
#     dest = us_ports[row["port_full"]]
#     return sr.searoute(origin, dest).properties["length"]


# df_asian_routes_wash["distance"] = df_asian_routes_wash.apply(compute_distance_ocean, axis=1)

In [26]:
df_asian_routes_wash_v1.describe()

,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,distance
count,7.200000e+01,72.000000,72.000000,7.200000e+01,7.200000e+01,7.200000e+01,7.200000e+01,72.000000
mean,1.101281e+06,83.333333,1.944444,1.101197e+06,2.816766e+05,1.100894e+06,2.816518e+05,15982.188424
std,1.825331e+06,707.106781,16.499158,1.825381e+06,5.138560e+05,1.825564e+06,5.138694e+05,4645.096364
min,3.888000e+03,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7005.745307
25%,5.163100e+04,0.000000,0.000000,5.163100e+04,1.222825e+04,5.163100e+04,1.160625e+04,10674.961095
50%,2.522800e+05,0.000000,0.000000,2.522800e+05,5.063850e+04,2.522800e+05,5.063850e+04,18277.800801
75%,1.254910e+06,0.000000,0.000000,1.254910e+06,2.900002e+05,1.254910e+06,2.900002e+05,19692.089414
max,8.275496e+06,6000.000000,140.000000,8.275496e+06,2.731801e+06,8.275496e+06,2.731801e+06,21740.992499


Because the other dataset(ISTHS6MM) does not have land transportation weight and only land transportation value, I will calculate average value of washing machines and use it to find land transportation weight for Mexico. Because there are no rows with 0 for vessel-transported value or vessel-transported weight, I do not need to worry about rows with 0s affecting the mean.

In [27]:
washing_machine_price_coeff = df_asian_routes_wash_v1["ves_val_mo"].mean()/df_asian_routes_wash_v1["ves_swt_mo"].mean()

print(washing_machine_price_coeff, "$/kg")

3.9094380055141054 $/kg


In [28]:
df_asian_routes_tv = df_asian_routes_v1[df_asian_routes_v1["commodity"].str.startswith("8528")]
if not df_asian_routes_tv.empty and df_asian_routes_tv["ves_swt_mo"].sum() > 0:
    tv_price_coeff = df_asian_routes_tv["ves_val_mo"].mean() / df_asian_routes_tv["ves_swt_mo"].mean()
else:
    tv_price_coeff = 15.0  # fallback: ~$15/kg for TVs
print(tv_price_coeff, "$/kg")

17.56019426681531 $/kg


The value of 3.91 $/kg makes sense, given that most washing machines weigh at around 50 kg and cost around 200 dollars.

In [29]:
df_asian_routes_wash_v1.to_csv("data/intermediate/PORTHS6MM_asian.csv")

## Land-based Distance/Weight Data (Mexico, Jan 2025) — DISABLED

This section previously processed `isthsm2501.txt`. Disabled: the DiD design compares Dec 2024 vs Dec 2025 (same month, different years).

In [30]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex = df_land_imports[df_land_imports["cty_code"] == "2010"]

In [31]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex.head()

In [32]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex_v1 = df_land_imports_mex.T.drop_duplicates().T
# df_land_imports_mex_v1.info()

In [33]:
# DISABLED (isthsm2501 Mexico pipeline)
# numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

# for col in numerical_cols:
#     df_land_imports_mex_v1[col] = pd.to_numeric(df_land_imports_mex_v1[col])
# df_land_imports_mex_v1.info()

In [34]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex_wash = df_land_imports_mex_v1[df_land_imports_mex_v1["commodity"].isin(["845011", "845020"])]
# df_land_imports_mex_wash.head()

So, upon filtering and cleaning data, we recognize that the vast majority of imports of Mexican washing machines into the USA is done through the land, rather than sea or plane. This confirmed my initial hypothesis on Mexico's transformation breakdown and now justifies the plan to calculate solely the inland leg's carbon footprint for imports from Mexico.

Now, we need to understand the distances from each state to state.

In [35]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

In [36]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex_wash_v1 = df_land_imports_mex_wash.copy()

# df_land_imports_mex_wash_v1["distance"] = df_land_imports_mex_wash_v1.apply(core.compute_distance, 
#     args = [coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

In [37]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex_wash_v1["gen_swt_mo"] = df_land_imports_mex_wash_v1["gen_val_mo"] / washing_machine_price_coeff

In [38]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex_wash_v1.info()

In [39]:
# DISABLED (isthsm2501 Mexico pipeline)
# df_land_imports_mex_wash_v1.to_csv("data/intermediate/ISTHS6MM_mex.csv")

## Land-based Distance/Weight Data (Mexico, Pre-Tariff — Dec 2024)

In [40]:
df_land_imports_mex_2412 = df_land_imports_2412[df_land_imports_2412["cty_code"] == "2010"]
df_land_imports_mex_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
31,010121,2010,AZ,2024,12,0,0,0,0,0,...,0,0,44862,44862,0,0,0,0,0,0
32,010121,2010,CA,2024,12,0,0,0,0,0,...,0,0,33250,33250,0,0,0,0,0,0
33,010121,2010,FL,2024,12,0,0,0,0,0,...,0,0,29000,29000,29000,3500,0,0,0,0
34,010121,2010,NM,2024,12,0,0,0,0,0,...,0,0,3000,3000,0,0,0,0,0,0
35,010121,2010,TX,2024,12,0,0,0,0,0,...,0,0,232003,232003,0,0,0,0,0,0


In [41]:
df_land_imports_mex_v1_2412 = df_land_imports_mex_2412.T.drop_duplicates().T
df_land_imports_mex_v1_2412.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 39825 entries, 31 to 1194457
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39825 non-null  object
 1   cty_code    39825 non-null  object
 2   state       39825 non-null  object
 3   year        39825 non-null  object
 4   month       39825 non-null  object
 5   gen_val_mo  39825 non-null  object
 6   con_val_mo  39825 non-null  object
 7   air_val_mo  39825 non-null  object
 8   air_swt_mo  39825 non-null  object
 9   ves_val_mo  39825 non-null  object
 10  ves_swt_mo  39825 non-null  object
 11  cnt_val_mo  39825 non-null  object
 12  cnt_swt_mo  39825 non-null  object
 13  gen_val_yr  39825 non-null  object
 14  con_val_yr  39825 non-null  object
 15  air_val_yr  39825 non-null  object
 16  air_swt_yr  39825 non-null  object
 17  ves_val_yr  39825 non-null  object
 18  ves_swt_yr  39825 non-null  object
 19  cnt_val_yr  39825 non-null  object
 20  cnt

In [42]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1_2412[col] = pd.to_numeric(df_land_imports_mex_v1_2412[col])
df_land_imports_mex_v1_2412.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 39825 entries, 31 to 1194457
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39825 non-null  object
 1   cty_code    39825 non-null  object
 2   state       39825 non-null  object
 3   year        39825 non-null  object
 4   month       39825 non-null  object
 5   gen_val_mo  39825 non-null  int64 
 6   con_val_mo  39825 non-null  int64 
 7   air_val_mo  39825 non-null  int64 
 8   air_swt_mo  39825 non-null  int64 
 9   ves_val_mo  39825 non-null  int64 
 10  ves_swt_mo  39825 non-null  int64 
 11  cnt_val_mo  39825 non-null  int64 
 12  cnt_swt_mo  39825 non-null  int64 
 13  gen_val_yr  39825 non-null  object
 14  con_val_yr  39825 non-null  object
 15  air_val_yr  39825 non-null  object
 16  air_swt_yr  39825 non-null  object
 17  ves_val_yr  39825 non-null  object
 18  ves_swt_yr  39825 non-null  object
 19  cnt_val_yr  39825 non-null  object
 20  cnt

In [43]:
df_land_imports_mex_wash_2412 = df_land_imports_mex_v1_2412[df_land_imports_mex_v1_2412["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
799500,845011,2010,CA,2024,12,2805860,2805860,0,0,0,...,0,0,28767168,28767168,0,0,40635,35100,40635,35100
799501,845011,2010,CO,2024,12,430796,430796,0,0,0,...,0,0,5790782,5790782,0,0,0,0,0,0
799502,845011,2010,FL,2024,12,1334984,1334984,0,0,328736,...,328736,73407,19850931,19850931,0,0,3583492,831963,3583492,831963
799503,845011,2010,GA,2024,12,1199977,1199977,0,0,0,...,0,0,16540032,16540032,0,0,0,0,0,0
799504,845011,2010,IN,2024,12,1157949,1157949,0,0,0,...,0,0,13289441,13289441,0,0,0,0,0,0


In [44]:
df_land_imports_mex_wash_2412["state"].value_counts()

CA    2
MD    2
WA    2
CO    2
PR    2
OH    2
TX    2
KY    2
IN    2
GA    2
FL    2
NC    1
AZ    1
IL    1
MA    1
MI    1
MN    1
MO    1
NE    1
NJ    1
PA    1
Name: state, dtype: int64

In [45]:
df_land_imports_mex_wash_v1_2412 = df_land_imports_mex_wash_2412.copy()

df_land_imports_mex_wash_v1_2412["distance"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

In [46]:
df_land_imports_mex_wash_v1_2412["gen_swt_mo"] = df_land_imports_mex_wash_v1_2412["gen_val_mo"] / washing_machine_price_coeff

In [47]:
df_land_imports_mex_wash_v1_2412["co2"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_land_imports_mex_wash_v1_2412.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 32 entries, 799500 to 799719
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   32 non-null     object 
 1   cty_code    32 non-null     object 
 2   state       32 non-null     object 
 3   year        32 non-null     object 
 4   month       32 non-null     object 
 5   gen_val_mo  32 non-null     int64  
 6   con_val_mo  32 non-null     int64  
 7   air_val_mo  32 non-null     int64  
 8   air_swt_mo  32 non-null     int64  
 9   ves_val_mo  32 non-null     int64  
 10  ves_swt_mo  32 non-null     int64  
 11  cnt_val_mo  32 non-null     int64  
 12  cnt_swt_mo  32 non-null     int64  
 13  gen_val_yr  32 non-null     object 
 14  con_val_yr  32 non-null     object 
 15  air_val_yr  32 non-null     object 
 16  air_swt_yr  32 non-null     object 
 17  ves_val_yr  32 non-null     object 
 18  ves_swt_yr  32 non-null     object 
 19  cnt_val_yr  32 non-nul

In [48]:
df_land_imports_mex_wash_v1_2412.to_csv("data/intermediate/ISTHS6MM_mex_2412.csv")

## Land-based Distance/Weight Data (Mexico, Post-Tariff — Dec 2025)

In [49]:
df_land_imports_mex_2512 = df_land_imports_2512[df_land_imports_2512["cty_code"] == "2010"]
df_land_imports_mex_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
26,010121,2010,FL,2025,12,24000,24000,24000,2000,0,...,0,0,57000,57000,57000,5500,0,0,0,0
160,010129,2010,AZ,2025,12,0,0,0,0,0,...,0,0,275173,275173,0,0,0,0,0,0
161,010129,2010,CA,2025,12,0,0,0,0,0,...,0,0,504000,504000,504000,4500,0,0,0,0
162,010129,2010,FL,2025,12,51700,51700,51700,4000,0,...,0,0,143200,143200,143200,12957,0,0,0,0
163,010129,2010,NM,2025,12,0,0,0,0,0,...,0,0,195600,195600,0,0,0,0,0,0


In [50]:
df_land_imports_mex_v1_2512 = df_land_imports_mex_2512.T.drop_duplicates().T
df_land_imports_mex_v1_2512.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 39926 entries, 26 to 1247779
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39926 non-null  object
 1   cty_code    39926 non-null  object
 2   state       39926 non-null  object
 3   year        39926 non-null  object
 4   month       39926 non-null  object
 5   gen_val_mo  39926 non-null  object
 6   con_val_mo  39926 non-null  object
 7   air_val_mo  39926 non-null  object
 8   air_swt_mo  39926 non-null  object
 9   ves_val_mo  39926 non-null  object
 10  ves_swt_mo  39926 non-null  object
 11  cnt_val_mo  39926 non-null  object
 12  cnt_swt_mo  39926 non-null  object
 13  gen_val_yr  39926 non-null  object
 14  con_val_yr  39926 non-null  object
 15  air_val_yr  39926 non-null  object
 16  air_swt_yr  39926 non-null  object
 17  ves_val_yr  39926 non-null  object
 18  ves_swt_yr  39926 non-null  object
 19  cnt_val_yr  39926 non-null  object
 20  cnt

In [51]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1_2512[col] = pd.to_numeric(df_land_imports_mex_v1_2512[col])
df_land_imports_mex_v1_2512.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 39926 entries, 26 to 1247779
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39926 non-null  object
 1   cty_code    39926 non-null  object
 2   state       39926 non-null  object
 3   year        39926 non-null  object
 4   month       39926 non-null  object
 5   gen_val_mo  39926 non-null  int64 
 6   con_val_mo  39926 non-null  int64 
 7   air_val_mo  39926 non-null  int64 
 8   air_swt_mo  39926 non-null  int64 
 9   ves_val_mo  39926 non-null  int64 
 10  ves_swt_mo  39926 non-null  int64 
 11  cnt_val_mo  39926 non-null  int64 
 12  cnt_swt_mo  39926 non-null  int64 
 13  gen_val_yr  39926 non-null  object
 14  con_val_yr  39926 non-null  object
 15  air_val_yr  39926 non-null  object
 16  air_swt_yr  39926 non-null  object
 17  ves_val_yr  39926 non-null  object
 18  ves_swt_yr  39926 non-null  object
 19  cnt_val_yr  39926 non-null  object
 20  cnt

In [52]:
df_land_imports_mex_wash_2512 = df_land_imports_mex_v1_2512[
    df_land_imports_mex_v1_2512["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
849178,845011,2010,CA,2025,12,1807730,1807730,0,0,0,...,0,0,27563248,27563248,0,0,0,0,0,0
849179,845011,2010,CO,2025,12,398827,398827,0,0,0,...,0,0,4922868,4922868,0,0,0,0,0,0
849180,845011,2010,FL,2025,12,1378826,1378826,0,0,0,...,0,0,19121576,19121576,0,0,254100,58590,254100,58590
849181,845011,2010,GA,2025,12,764299,764299,0,0,0,...,0,0,13078900,13078900,0,0,0,0,0,0
849182,845011,2010,IL,2025,12,716040,716040,0,0,0,...,0,0,8086170,8086170,0,0,0,0,0,0


In [53]:
df_land_imports_mex_wash_v1_2512 = df_land_imports_mex_wash_2512.copy()

df_land_imports_mex_wash_v1_2512["distance"] = df_land_imports_mex_wash_v1_2512.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_wash_v1_2512["gen_swt_mo"] = (
    df_land_imports_mex_wash_v1_2512["gen_val_mo"] / washing_machine_price_coeff)

df_land_imports_mex_wash_v1_2512["co2"] = df_land_imports_mex_wash_v1_2512.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_wash_v1_2512.to_csv("data/intermediate/ISTHS6MM_mex_2512.csv")
df_land_imports_mex_wash_v1_2512.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 29 entries, 849178 to 849386
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   29 non-null     object 
 1   cty_code    29 non-null     object 
 2   state       29 non-null     object 
 3   year        29 non-null     object 
 4   month       29 non-null     object 
 5   gen_val_mo  29 non-null     int64  
 6   con_val_mo  29 non-null     int64  
 7   air_val_mo  29 non-null     int64  
 8   air_swt_mo  29 non-null     int64  
 9   ves_val_mo  29 non-null     int64  
 10  ves_swt_mo  29 non-null     int64  
 11  cnt_val_mo  29 non-null     int64  
 12  cnt_swt_mo  29 non-null     int64  
 13  gen_val_yr  29 non-null     object 
 14  con_val_yr  29 non-null     object 
 15  air_val_yr  29 non-null     object 
 16  air_swt_yr  29 non-null     object 
 17  ves_val_yr  29 non-null     object 
 18  ves_swt_yr  29 non-null     object 
 19  cnt_val_yr  29 non-nul

## Land-based Distance/Weight Data (Mexico, Control — HS 8528 TVs, Dec 2024)

In [54]:
df_land_imports_mex_tv_2412 = df_land_imports_mex_v1_2412[
    df_land_imports_mex_v1_2412["commodity"].str.startswith("8528")]
df_land_imports_mex_tv_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
942710,852842,2010,PR,2024,12,0,0,0,0,0,...,0,0,136900,136900,136900,1350,0,0,0,0
942737,852849,2010,PR,2024,12,0,0,0,0,0,...,0,0,162288,162288,162288,1972,0,0,0,0
942847,852852,2010,AL,2024,12,0,0,0,0,0,...,0,0,2290,2290,0,0,0,0,0,0
942848,852852,2010,AZ,2024,12,0,0,0,0,0,...,0,0,16250,16250,14000,297,0,0,0,0
942849,852852,2010,CA,2024,12,3059946,3059946,0,0,0,...,0,0,60472486,60472486,551557,1770,0,0,0,0


In [55]:
df_land_imports_mex_tv_2412[df_land_imports_mex_tv_2412["state"]=="VI"]

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
945083,852872,2010,VI,2024,12,0,0,0,0,0,...,0,0,8442,8442,0,0,8442,29,0,0


In [56]:
df_land_imports_mex_tv_v1_2412 = df_land_imports_mex_tv_2412.copy()

# I did this, because it is logically impossible for commodities to be imported into the US via the Virgin Islands from Mexico, so this is likely a data error. 
# I will exclude it from the analysis.
df_land_imports_mex_tv_v1_2412 = df_land_imports_mex_tv_v1_2412.drop(df_land_imports_mex_tv_v1_2412[df_land_imports_mex_tv_v1_2412["state"]=="VI"].index, inplace=False)

df_land_imports_mex_tv_v1_2412["distance"] = df_land_imports_mex_tv_v1_2412.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_tv_v1_2412["gen_swt_mo"] = (
    df_land_imports_mex_tv_v1_2412["gen_val_mo"] / tv_price_coeff)

df_land_imports_mex_tv_v1_2412["co2"] = df_land_imports_mex_tv_v1_2412.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_tv_v1_2412.to_csv("data/intermediate/ISTHS6MM_mex_tv_2412.csv")
df_land_imports_mex_tv_v1_2412.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 112 entries, 942710 to 945085
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   112 non-null    object 
 1   cty_code    112 non-null    object 
 2   state       112 non-null    object 
 3   year        112 non-null    object 
 4   month       112 non-null    object 
 5   gen_val_mo  112 non-null    int64  
 6   con_val_mo  112 non-null    int64  
 7   air_val_mo  112 non-null    int64  
 8   air_swt_mo  112 non-null    int64  
 9   ves_val_mo  112 non-null    int64  
 10  ves_swt_mo  112 non-null    int64  
 11  cnt_val_mo  112 non-null    int64  
 12  cnt_swt_mo  112 non-null    int64  
 13  gen_val_yr  112 non-null    object 
 14  con_val_yr  112 non-null    object 
 15  air_val_yr  112 non-null    object 
 16  air_swt_yr  112 non-null    object 
 17  ves_val_yr  112 non-null    object 
 18  ves_swt_yr  112 non-null    object 
 19  cnt_val_yr  112 non-n

## Land-based Distance/Weight Data (Mexico, Control — HS 8528 TVs, Dec 2025)

In [57]:
df_land_imports_mex_tv_2512 = df_land_imports_mex_v1_2512[
    df_land_imports_mex_v1_2512["commodity"].str.startswith("8528")]
df_land_imports_mex_tv_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
992995,852849,2010,CA,2025,12,0,0,0,0,0,...,0,0,28133,28133,0,0,0,0,0,0
993095,852852,2010,AL,2025,12,0,0,0,0,0,...,0,0,20107,20107,0,0,0,0,0,0
993096,852852,2010,AR,2025,12,74797,74797,0,0,0,...,0,0,141945,141945,0,0,0,0,0,0
993097,852852,2010,AZ,2025,12,0,0,0,0,0,...,0,0,38688,38688,0,0,0,0,0,0
993098,852852,2010,CA,2025,12,25343683,25343683,0,0,0,...,0,0,193420909,193420909,119413,279,0,0,0,0


In [58]:
df_land_imports_mex_tv_v1_2512 = df_land_imports_mex_tv_2512.copy()

# I did this, because it is logically impossible for commodities to be imported into the US via the Virgin Islands from Mexico, so this is likely a data error. 
# I will exclude it from the analysis.
df_land_imports_mex_tv_v1_2512 = df_land_imports_mex_tv_v1_2512.drop(df_land_imports_mex_tv_v1_2512[df_land_imports_mex_tv_v1_2512["state"]=="VI"].index, inplace=False)


df_land_imports_mex_tv_v1_2512["distance"] = df_land_imports_mex_tv_v1_2512.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_tv_v1_2512["gen_swt_mo"] = (
    df_land_imports_mex_tv_v1_2512["gen_val_mo"] / tv_price_coeff)

df_land_imports_mex_tv_v1_2512["co2"] = df_land_imports_mex_tv_v1_2512.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_tv_v1_2512.to_csv("data/intermediate/ISTHS6MM_mex_tv_2512.csv")
df_land_imports_mex_tv_v1_2512.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 124 entries, 992995 to 995325
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   124 non-null    object 
 1   cty_code    124 non-null    object 
 2   state       124 non-null    object 
 3   year        124 non-null    object 
 4   month       124 non-null    object 
 5   gen_val_mo  124 non-null    int64  
 6   con_val_mo  124 non-null    int64  
 7   air_val_mo  124 non-null    int64  
 8   air_swt_mo  124 non-null    int64  
 9   ves_val_mo  124 non-null    int64  
 10  ves_swt_mo  124 non-null    int64  
 11  cnt_val_mo  124 non-null    int64  
 12  cnt_swt_mo  124 non-null    int64  
 13  gen_val_yr  124 non-null    object 
 14  con_val_yr  124 non-null    object 
 15  air_val_yr  124 non-null    object 
 16  air_swt_yr  124 non-null    object 
 17  ves_val_yr  124 non-null    object 
 18  ves_swt_yr  124 non-null    object 
 19  cnt_val_yr  124 non-n

## Weight Data(USA Trade Online) - irrelevant?

Already well-organized, the weight data does not need much cleaning.

However, the previous datasets(US Census) contains everything necessary. Not just weight of imports, but also origin and destination info. Use this as a sanity check for weight of ocean-side imports.

In [59]:
washing_machine_df_v1 = washing_machine_df.drop("Unnamed: 5", axis=1)
washing_machine_df_v1["Air SWT (kg)"] = washing_machine_df_v1["Air SWT (kg)"].str.replace(',', '')
washing_machine_df_v1["Vessel SWT (kg)"] = washing_machine_df_v1["Vessel SWT (kg)"].str.replace(',', '')

washing_machine_df_v1["Air SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Air SWT (kg)"])
washing_machine_df_v1["Vessel SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Vessel SWT (kg)"])

washing_machine_df_v1 = washing_machine_df_v1.fillna(0)

In [60]:
washing_machine_df_v1.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174.0,1777528.0
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,0.0,2459827.0
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,0.0,2268490.0
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,0.0,2512291.0
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,0.0,2541012.0


In [61]:
washing_machine_df_v1[washing_machine_df_v1["Country"]=="Mexico"].head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
32,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,January 2025,0.0,55246.0
33,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,February 2025,0.0,74445.0
34,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,March 2025,0.0,239852.0
35,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,April 2025,0.0,175512.0
36,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,May 2025,0.0,199598.0


In [62]:
washing_machine_df_v1["Time"].value_counts()

April 2025            9
June 2025             9
July 2025             9
October 2025          9
December 2025         9
2026 through March    9
January 2026          9
January 2025          8
February 2025         8
March 2025            8
May 2025              8
August 2025           8
September 2025        8
November 2025         8
February 2026         8
March 2026            8
Name: Time, dtype: int64

In [63]:
washing_machine_df_v1["Air SWT (kg)"].mean()

171.28148148148148

# CO2 Calculations(transportation)

Now that we have the three necessary components - ton-km CO2 factor, weight of imports and distance of imports, then we can calculate the carbon footprint of transportation. This is only one step, as we also need to calculate CO2 of manufacturing(grid intensity x energy spent on manufacturing) in order to understand the full extent of the carbon footprint.

CO2, in this case, is measured in grams. For now, we do this only for 5 countries(China, Vietnam, South Korea, India, Mexico) and only for the month of January 2025, using the datasets taken from census.gov website.

In [64]:
# def compute_co2(row, transportation_type):
#     weight = row[transportation_type]
#     dist = row["distance"]
#     co2_coeff = ton_km[row["cty_code"]]
#     return weight * dist * co2_coeff

df_land_imports_mex_wash_v1_2412["co2"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_co2, args = ["gen_swt_mo"], axis=1)

In [65]:
df_land_imports_mex_wash_v1_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,distance,gen_swt_mo,co2
799500,845011,2010,CA,2024,12,2805860,2805860,0,0,0,...,28767168,0,0,40635,35100,40635,35100,2847.115733,717714.412159,1.634733e+11
799501,845011,2010,CO,2024,12,430796,430796,0,0,0,...,5790782,0,0,0,0,0,0,2028.894380,110193.843563,1.788573e+10
799502,845011,2010,FL,2024,12,1334984,1334984,0,0,328736,...,19850931,0,0,3583492,831963,3583492,831963,2442.374793,341477.214402,6.672123e+10
799503,845011,2010,GA,2024,12,1199977,1199977,0,0,0,...,16540032,0,0,0,0,0,0,2389.453884,306943.606295,5.867421e+10
799504,845011,2010,IN,2024,12,1157949,1157949,0,0,0,...,13289441,0,0,0,0,0,0,2660.910286,296193.212008,6.305149e+10


In [66]:
df_land_imports_mex_wash_v1_2412.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 32 entries, 799500 to 799719
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   32 non-null     object 
 1   cty_code    32 non-null     object 
 2   state       32 non-null     object 
 3   year        32 non-null     object 
 4   month       32 non-null     object 
 5   gen_val_mo  32 non-null     int64  
 6   con_val_mo  32 non-null     int64  
 7   air_val_mo  32 non-null     int64  
 8   air_swt_mo  32 non-null     int64  
 9   ves_val_mo  32 non-null     int64  
 10  ves_swt_mo  32 non-null     int64  
 11  cnt_val_mo  32 non-null     int64  
 12  cnt_swt_mo  32 non-null     int64  
 13  gen_val_yr  32 non-null     object 
 14  con_val_yr  32 non-null     object 
 15  air_val_yr  32 non-null     object 
 16  air_swt_yr  32 non-null     object 
 17  ves_val_yr  32 non-null     object 
 18  ves_swt_yr  32 non-null     object 
 19  cnt_val_yr  32 non-nul

In [67]:
# df_asian_routes_wash["co2"] = df_asian_routes_wash["vessel_swt_mo"]*df_asian_routes_wash["distance"]*ton_km[df_asian_routes_wash["cty_code"]]
df_asian_routes_wash_v1["co2"] = df_asian_routes_wash_v1.apply(core.compute_co2, args = ["ves_swt_mo"], axis=1)


In [68]:
df_asian_routes_wash_v1.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,ves_swt_mo,cnt_val_mo,cnt_swt_mo,port_full,distance,co2
273372,845011,5520,10,03,2025,01,1476672,0,0,1476672,215096,1476672,215096,1003,20027.006554,3.015410e+10
273373,845011,5520,13,03,2025,01,3888,0,0,3888,740,3888,740,1303,20589.054455,1.066513e+08
273374,845011,5520,14,01,2025,01,8976,0,0,8976,2224,8976,2224,1401,20353.933792,3.168700e+08
273375,845011,5520,17,03,2025,01,153763,0,0,153763,21411,153763,21411,1703,20949.030244,3.139778e+09
273376,845011,5520,27,04,2025,01,19801,0,0,19801,4583,19801,4583,2704,13525.167789,4.339009e+08


In [69]:
df_asian_routes_wash_v1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 72 entries, 273372 to 273506
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    72 non-null     object 
 1   cty_code     72 non-null     object 
 2   dist_unlade  72 non-null     object 
 3   port_unlade  72 non-null     object 
 4   year         72 non-null     object 
 5   month        72 non-null     object 
 6   gen_val_mo   72 non-null     int64  
 7   air_val_mo   72 non-null     int64  
 8   air_swt_mo   72 non-null     int64  
 9   ves_val_mo   72 non-null     int64  
 10  ves_swt_mo   72 non-null     int64  
 11  cnt_val_mo   72 non-null     int64  
 12  cnt_swt_mo   72 non-null     int64  
 13  port_full    72 non-null     object 
 14  distance     72 non-null     float64
 15  co2          72 non-null     float64
dtypes: float64(2), int64(7), object(7)
memory usage: 9.6+ KB


# Statistics

Now we use the difference-in-difference method to evaluate the extent, to which the tariffs have changed the CO2 footprint. We compare the differences in the control group and the treatment group. 

What will be the control group in our case? 

The control group will be "commodity", more specifically TV sets. TVs have mostly the same demand as washing machines, since both are bought together when people move to a new home. However, unlike the washing machines, TV sets don't fall under the Liberation Day tariffs and steel tariffs of 2025, which is due to goods with semiconductors being exempt from tariffs. These factors make TVs a good control group, since they're similar to washing machines in most key regards, except for the thing that we try to measure.

However, TV sets are lighter than washing machines, so CO2 change scales differently. One way to fix this is to normalize CO2 data by calculating CO2/kg of commodity, rather than total CO2. 


First, we must assemble an econometric model.

Let us start with a basic 2x2 regression equation:

$Y_i = \alpha + \beta*ifTariffShock\_c*\delta*PostTariff\_t+\gamma*(ifTariffShock*PostTariff)+\epsilon_i$

+ $Y_i$ represents grams of CO2 emissions of supply chains of imports from a particular country $c$ at a certain time $t$
+ ifTariffShock_c is a boolean that represents if a country has been affected by the tariff shock. 1 if it is, 0 if not.
+ PostTariff_t is a boolean that represents if the observation takes place before or after the tariff. 0 if the date is Dec 2024, 1 if it's Dec 2025
+ $\gamma$ is the difference-in-differences estimator.

For the model, we need the following assumptions:
+ Parallel trends
    + Flaaen, Hortaçsu & Tintelnot (2020) did this by comparing the trends of the treatment group(washing machines) with the control group(refrigerators, dishwashes and other un-tariffed appliances) before the tariffs.
    + We do this because we cannot see what the countries would've done WITHOUT tariffs. However, we CAN test whether these groups moved in parallel before the tariff.
    + Pre-trend plot necessary to demonstrate it?
+ Anticipation
    + Expectations shape economics. If the importers already knew that the tariffs would've happened, this would've influenced their behavior compared to if they didn't know about the tariffs beforehand.
    + Announcements come many weeks before the actual tariff, as evidenced by Freund et al. (2024). 
+ Error correlates over time within a country
    + How do we address that?
+ Seasonality
    + We assume that washing machine demand is affected by seasons. To avoid the error caused by that, we compare the same month of a different year(Dec 2024 and Dec 2025) respectively. 
    + The announcement date for Liberation Day Tariffs was Feb 13th 2025, so this covers the "anticipation" assumption
+ Semiconductor tariff exception for TV sets is valid
    + While the language around the tariff exception for semiconductors doesn't clarify(rewrite? how exactly is the ambiguity problematic?) the status of TV sets, we assume that they fall under that exemption, and that every party in the supply chain of TVs recognizes that.

This code sets up a DiD model and runs a OLS regression for Mexico only. This is purely for testing purposes. Later on, I might think of using a different regression model and adding more countries to the list. 

In [70]:
import statsmodels.api as sm

def prep_did(df, treated_flag, post_flag):
    agg = df.groupby("state").agg(
        co2=("co2", "sum"),
        gen_val_mo=("gen_val_mo", "sum")
    ).reset_index()
    agg["treated"] = treated_flag
    agg["post"] = post_flag
    return agg

# prepares the data for difference-in-difference analysis. 
# treated_flag represents whether the group is treated (1 for washing machines, 0 for TVs)
# post_flag represents whether the data is from after the tariff implementation (1 for 2025, 0 for 2024).
did_parts = [
    prep_did(df_land_imports_mex_wash_v1_2412, treated_flag=1, post_flag=0),
    prep_did(df_land_imports_mex_wash_v1_2512, treated_flag=1, post_flag=1),
    prep_did(df_land_imports_mex_tv_v1_2412,   treated_flag=0, post_flag=0),
    prep_did(df_land_imports_mex_tv_v1_2512,   treated_flag=0, post_flag=1),
]

# concatenates the difference-in-difference parts into one
df_did = pd.concat(did_parts, ignore_index=True)
df_did["treated_x_post"] = df_did["treated"] * df_did["post"]
df_did.sort_values("co2", ascending=False).head(10)

,state,co2,gen_val_mo,treated,post,treated_x_post
86,CA,7.828767e+12,603569881,0,1,0
42,CA,4.245075e+12,327280076,0,0,0
47,GA,1.332961e+12,122449841,0,0,0
91,GA,1.087278e+12,99880615,0,1,0
65,NJ,5.972340e+11,34938716,0,0,0
37,TX,4.102752e+11,23509099,1,1,1
19,TX,3.686873e+11,21126079,1,0,0
22,CA,3.224782e+11,5535025,1,1,1
120,TX,2.944216e+11,75778379,0,1,0
9,MD,2.597324e+11,3684188,1,0,0


It makes sense for carbon footprint of imports to California, Georgia and Texas to be high, since those states border Mexico. However, why do imports of TVs from Mexico to New Jersey have such high total value and CO2 footprint?

In [71]:
X = sm.add_constant(df_did[["treated", "post", "treated_x_post"]])
model = sm.OLS(df_did["co2"], X).fit(cov_type="HC3")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    co2   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                 -0.018
Method:                 Least Squares   F-statistic:                    0.5532
Date:                Fri, 19 Jun 2026   Prob (F-statistic):              0.647
Time:                        11:45:09   Log-Likelihood:                -3660.0
No. Observations:                 127   AIC:                             7328.
Df Residuals:                     123   BIC:                             7339.
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           1.524e+11   1.02e+11      1.

Having run the OLS regression on the difference-in-differences model, we found that for Mexico's imports, there is no difference in the CO2 before and after the tariffs. 

Why could that be? I know that for land transportation to the USA, the number of trade routes is more limited compared to ocean routes, so there is little that Mexico can do to change their trade routes in the aftermath of tariffs. But how much did the number of imported goods change?

Also, how do I interpret those other statistics? F-statistic, log-likelihood, AIC/BIC, Durbin-Watson, et cetera.